# v2 overview

**What v1a is.** The team's work from v1 and the open branches, organized as one folder per pipeline component.
Each folder has a notebook that is updated in place, its scripts and tests, and a matching output folder
(`outputs-local/v1a/<component>/`). `v1/` stays frozen as the record. One file, `v2/current.toml`, names every
current setting and saved run; the notebooks read it, so they agree by construction.

**How to follow it.** Read this page, then the notebooks in order. Each one has the same sections: what it does
(with a changelog), the current setting, its inputs (with a check against the v1 seed), results, checks, and history.
The checks fail loudly, so a notebook that runs to the end is aligned with `current.toml`.

**Rules.**
- The holdout (160 companies; Barbara's 24 direct-peer pairs) is the locked final check: only `10_final_holdout`
  reads it, and only with `FINAL_CHECK = True`.
- EV/EBITDA is the answer key: only `08_valuation` and `10_final_holdout` read the financials.
- Claude fills profiles and selects comps; GPT models judge (evaluation only).
- New LLM runs (CLI subscriptions) need approval first.

**Changelog**
- 2026-11-19: created. All ten components and the runner are ported.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v2" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v2" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

## 1. The pipeline

In [2]:
import json, re, glob
NOTEBOOKS = {"01_universe": "universe", "02_profiles": "profiles", "03_retrieval": "retrieval", "04_selection": "selection",
             "05_agent_grading": "agent_grading", "06_labels": "labels", "07_judges": "judges", "08_valuation": "valuation",
             "09_analysis": "analysis", "10_final_holdout": "final_holdout", "pipeline": "pipeline"}
WHAT = {"01_universe": "533 companies; 373 training / 160 holdout (SIC2 split)",
        "02_profiles": "Sonnet facet fills + Fiona's sector/sub-sector + industry facets",
        "03_retrieval": "B3 per-facet similarity, mutual top 10 topped up to 3",
        "04_selection": "Sonnet selector keeps or drops each B3 comp",
        "05_agent_grading": "agent loop grades pairs 0/1/2 with citations; facet scores; C/S/E ranking",
        "06_labels": "people's grades, cohorts, the pair ledger",
        "07_judges": "GPT judges grade pairs (evaluation only)",
        "08_valuation": "median comp EV/EBITDA vs actual",
        "09_analysis": "X-learner facet lift; error analysis",
        "10_final_holdout": "locked final check",
        "pipeline": "Kethan's end-to-end runner (fresh run; preview by default)"}

def last_stdout(path):
    nbk = json.loads(path.read_text(encoding="utf-8"))
    outs = [o.get("text", "") for c in nbk["cells"] if c["cell_type"] == "code" for o in c.get("outputs", []) if o.get("name") == "stdout"]
    text = "".join(outs[-1]) if outs else ""
    return text.strip().splitlines()[-1] if text.strip() else ""

rows = []
for comp, name in NOTEBOOKS.items():
    folder = ROOT / "v2" / comp
    nbpath = folder / f"{name}.ipynb"
    rows.append({"component": comp, "what": WHAT[comp], "notebook": f"{name}.ipynb",
                 "scripts": len(list((folder / "scripts").glob("*.py"))), "tests": len(list((folder / "tests").glob("test_*.py"))),
                 "last saved check": last_stdout(nbpath) if nbpath.exists() else "missing"})
status = pd.DataFrame(rows)
status

,component,what,notebook,scripts,tests,last saved check
0,01_universe,533 companies; 373 training / 160 holdout (SIC2 split),universe.ipynb,2,1,7/7 checks pass
1,02_profiles,Sonnet facet fills + Fiona's sector/sub-sector + industry facets,profiles.ipynb,12,3,7/7 checks pass
2,03_retrieval,"B3 per-facet similarity, mutual top 10 topped up to 3",retrieval.ipynb,7,4,6/6 checks pass
3,04_selection,Sonnet selector keeps or drops each B3 comp,selection.ipynb,5,3,6/6 checks pass
4,05_agent_grading,agent loop grades pairs 0/1/2 with citations; facet scores; C/S/E ranking,agent_grading.ipynb,5,4,5/5 checks pass
5,06_labels,"people's grades, cohorts, the pair ledger",labels.ipynb,10,6,7/7 checks pass
6,07_judges,GPT judges grade pairs (evaluation only),judges.ipynb,3,0,5/5 checks pass
7,08_valuation,median comp EV/EBITDA vs actual,valuation.ipynb,2,1,5/5 checks pass
8,09_analysis,X-learner facet lift; error analysis,analysis.ipynb,3,2,14/14 checks pass
9,10_final_holdout,locked final check,final_holdout.ipynb,1,1,Locked. Set FINAL_CHECK = True when the team decides the method is final.


## 2. Current settings (`v2/current.toml`)

In [3]:
def flat(d, prefix=""):
    for k, v in d.items():
        if isinstance(v, dict):
            yield from flat(v, f"{prefix}{k}.")
        else:
            yield f"{prefix}{k}", v
pd.Series(dict(flat(cur))).to_frame("value")

,value
universe.snapshot,company_list_v3
universe.universe_run,20261025T060358Z
universe.split_run,20261027T204930Z
universe.universe,outputs-local/v1a/01_universe/universe/universe.csv
universe.split,outputs-local/v1a/01_universe/split/split.csv
profiles.general,outputs-local/v1a/02_profiles/facets_sonnet/tables/facets.csv
profiles.general_prompt_version,2026-10-25k
profiles.sector,outputs-local/v1a/02_profiles/sector_facets_sonnet/tables/sector_facets.csv
profiles.industry,outputs-local/v1a/02_profiles/industry_facets_sonnet/tables/industry_facets_long.csv
profiles.corrected_general,outputs-local/v1a/02_profiles/facets_sonnet_corrected.csv


## 3. Pairs judged through the pipeline (06 pair ledger)

Every target-comp pair a person or a model judged, by stage and by rubric or prompt version (Kethan's ask). Pairs
whose comp is outside the universe are left out.

In [4]:
ledger = paths.latest("06_labels", "pair_ledger")
print("ledger run:", ledger.name)
spans = pd.read_csv(ledger / "spans.csv")
spans[["stage", "rubric", "who", "first", "last", "judgments", "unique_pairs_judged", "targets"]]

ledger run: 20261119T014543Z


,stage,rubric,who,first,last,judgments,unique_pairs_judged,targets
0,llm_screen,binary screen prompt 2026-10-25a,claude-haiku-5-5,2026-10-27,2026-10-27,4996,4996,373
1,human_review,"round 1: 0/1/2, blank = 0 with one of five reasons",Barbara;Matthew;Riya,2026-10-28,2026-10-28,108,36,2
2,llm_judge,"cited grades, 0/1/2 (v0 holdout)",GPT-6.1 Sol,2026-10-28,2026-10-28,1982,1982,156
3,agent_loop,agent prompt 2026-11-05a,sonnet agent,2026-11-05,2026-11-06,505,341,47
4,agent_loop,agent prompt 2026-11-05b,sonnet agent,2026-11-05,2026-11-05,146,71,7
5,llm_screen,binary screen prompt 2026-11-04a,claude-haiku-5-5;claude-sonnet-5-5,2026-11-05,2026-11-19,7330,3148,373
6,human_review,"rounds 2-3 old form: 0/1/2, six reasons, multi-business rule",Barbara,2026-11-10,2026-11-10,293,293,45
7,llm_judge,"source-cited one-shot judge, 0/1/2",GPT-6.1 Sol,2026-11-10,2026-11-10,293,293,45
8,agent_loop,agent prompt 2026-11-05b + facet scores,sonnet agent,2026-11-12,2026-11-18,502,313,45
9,human_review,"rubric v2: four yes/no questions, score derived",Matthew,2026-11-13,2026-11-13,262,262,40


## 4. The current run, in numbers

In [5]:
split = pd.read_csv(paths.path(cur["universe"]["split"]), dtype=str)
top = pd.read_csv(paths.path(cur["retrieval"]["candidates"]) / "candidates.csv", dtype={"target": str, "comp": str})
j = pd.read_csv(paths.path(cur["selection"]["judgments"]), dtype={"target": str, "comp": str})
v = pd.read_csv(paths.path(cur["agent_grading"]["run"]) / "verdicts.csv", dtype={"target": str, "comp": str})
import cohorts
pd.Series({"universe": len(split), "training targets (and comp pool)": int((split["sample"] == "training").sum()),
           "holdout companies (locked)": int((split["sample"] == "holdout").sum()),
           "B3 pairs (current lists)": int(top["selected"].sum()),
           "selector kept": int((j["score"] == 1).sum()),
           "targets where the selector kept none": int(j.groupby("target")["score"].sum().eq(0).sum()),
           "agent-graded pairs (current run)": len(v),
           **{f"cohort {n}": len(cohorts.load(n)) for n in cur["labels"]["cohorts"]},
           "locked final-check pairs": 24}).to_frame("count")

,count
universe,533
training targets (and comp pool),373
holdout companies (locked),160
B3 pairs (current lists),2538
selector kept,507
targets where the selector kept none,108
agent-graded pairs (current run),293
cohort dev_293,293
cohort round1_36,36
cohort matthew_v2_262,262


## 5. Which facets matter (09 X-learner)

Lift in percentage points of a strong B3 match on each facet: on people's peer grade (`dev_293`) and on the Sonnet
selector's keep (all current B3 pairs). Details and caveats in `09_analysis/analysis.ipynb`.

In [6]:
import facet_uplift as fu
runs = {k: paths.path(p) for k, p in cur["analysis"]["facet_uplift"].items()}
t = fu.lift_table({"people (293)": runs["human_peer"], "selector (2538)": runs["selector_all"]})
(t.xs("lift", axis=1, level=1).loc["b3"] * 100).round(1).sort_values("people (293)", ascending=False)

,people (293),selector (2538)
facet,,
primary_services,23.8,6.0
end_markets,21.4,8.9
customer_segment,20.7,2.5
primary_products,15.1,9.4
revenue_model,8.9,7.9
regulatory_exposure,8.7,6.2
specialization,5.3,4.9
offering_description,4.1,11.3
sub_sector,2.5,13.2


## 6. Checks

In [7]:
for comp, name in NOTEBOOKS.items():
    s = status.set_index("component").loc[comp, "last saved check"]
    if comp == "pipeline":
        checks.add("pipeline notebook previews without model calls", s.startswith("Preview only"), s)
    elif comp == "10_final_holdout":
        checks.add("10 is locked (FINAL_CHECK = False)", s.startswith("Locked"), s)
    else:
        m = re.match(r"(\d+)/(\d+) checks pass", s)
        checks.add(f"{comp}: saved notebook ran all its checks", bool(m) and m.group(1) == m.group(2), s)
manifest = json.loads((paths.V1A_OUTPUTS / "seed_manifest.json").read_text())
checks.add("seed manifest present", bool(manifest.get("copied")) or bool(manifest.get("files")), "outputs-local/v1a/seed_manifest.json")
checks.add("every component has a README", all((ROOT / "v2" / c / "README.md").exists() for c in NOTEBOOKS))
checks.show()

13/13 checks pass


,check,result,detail
0,01_universe: saved notebook ran all its checks,pass,7/7 checks pass
1,02_profiles: saved notebook ran all its checks,pass,7/7 checks pass
2,03_retrieval: saved notebook ran all its checks,pass,6/6 checks pass
3,04_selection: saved notebook ran all its checks,pass,6/6 checks pass
4,05_agent_grading: saved notebook ran all its checks,pass,5/5 checks pass
5,06_labels: saved notebook ran all its checks,pass,7/7 checks pass
6,07_judges: saved notebook ran all its checks,pass,5/5 checks pass
7,08_valuation: saved notebook ran all its checks,pass,5/5 checks pass
8,09_analysis: saved notebook ran all its checks,pass,14/14 checks pass
9,10 is locked (FINAL_CHECK = False),pass,Locked. Set FINAL_CHECK = True when the team decides the method is final.
